# Task 12: Convolutional autoencoder anomaly detection with SSIM loss

In [1]:
import torch
import torch.nn as nn


In [2]:
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32*16*16, 64)
        )
        self.decoder_fc = nn.Linear(64, 32*16*16)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 4, stride=2, padding=1), nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        h = self.decoder_fc(z).view(-1, 32, 16, 16)
        return self.decoder(h)


In [3]:
def gaussian_window(size, sigma):
    coords = torch.arange(size).float() - size//2
    g = torch.exp(-(coords**2)/(2*sigma**2))
    g = g/g.sum()
    return g.unsqueeze(0)*g.unsqueeze(1)

def ssim_loss(img1, img2, window_size=11, sigma=1.5):
    window = gaussian_window(window_size, sigma).unsqueeze(0).unsqueeze(0)
    mu1 = torch.nn.functional.conv2d(img1, window, padding=window_size//2)
    mu2 = torch.nn.functional.conv2d(img2, window, padding=window_size//2)
    sigma1 = torch.nn.functional.conv2d(img1*img1, window, padding=window_size//2) - mu1**2
    sigma2 = torch.nn.functional.conv2d(img2*img2, window, padding=window_size//2) - mu2**2
    sigma12 = torch.nn.functional.conv2d(img1*img2, window, padding=window_size//2) - mu1*mu2
    C1, C2 = 0.01**2, 0.03**2
    ssim_map = ((2*mu1*mu2+C1)*(2*sigma12+C2)) / ((mu1**2+mu2**2+C1)*(sigma1+sigma2+C2))
    return 1 - ssim_map.mean()


In [4]:
model = ConvAutoencoder()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
normal_imgs = torch.rand(32, 1, 64, 64)  # fault-free images only

for epoch in range(5):
    opt.zero_grad()
    recon = model(normal_imgs)
    loss = ssim_loss(recon, normal_imgs)
    loss.backward()
    opt.step()
    print(epoch, loss.item())


0 0.9021819829940796
1 0.9001222848892212
2 0.894526481628418
3 0.8844801783561707
4 0.8689945340156555


In [5]:
test_img = torch.rand(1, 1, 64, 64)
test_img[:,:,20:30,20:30] = 1.0  # inject an anomaly patch

with torch.no_grad():
    recon = model(test_img)
    error_map = (recon - test_img).abs()

print("max reconstruction error:", error_map.max().item())


max reconstruction error: 0.816423773765564
